In [0]:
%run "./01_data_ingestion"

In [0]:
# ============================================================
# 02.3 VALIDAÇÃO DO DATAFRAME
# ============================================================

# O QUE FAZ:
# Valida se o DataFrame disponibilizado pela etapa de ingestão está disponível e possui estrutura suficiente para iniciar o processo de Data Quality.

# COMO FAZ:
# Verifica a existência do objeto df, a quantidade de colunas e a existência de registros no DataFrame.

# POR QUE É IMPORTANTE:
# Evita que os Profiles seguintes sejam executados sobre um DataFrame inexistente ou estruturalmente inválido.

# PERGUNTA RESPONDIDA:
# "O DataFrame está disponível e possui estrutura válida?"


if "df" not in locals():
    raise ValueError(
        "O Dataframe 'df' não foi disponibilizado pelo Data Ingestion"
    )

if len(df.columns) == 0:
    raise ValueError(
        "O Dataframe não possui colunas"
    )

total_colunas = len(df.columns)
total_registers = df.count()

if total_registers == 0:
    raise ValueError(
        f"O Dataframe possui {total_colunas} colunas, mas não possui nenhum registro"
    )

print("Dataframe validado com sucesso")
print(f"Total de colunas: {total_colunas}")
print(f"Total de registros: {total_registers}")


In [0]:
# ============================================================
# 02.4 IDENTIFICAÇÃO DO SCHEMA
# ============================================================

# O QUE FAZ:
# Identifica automaticamente o schema do DataFrame e organiza as informações estruturais de cada coluna.

# COMO FAZ:
# Percorre o schema Spark e registra nome, tipo de dado e posição de cada coluna.

# POR QUE É IMPORTANTE:
# Permite que os Profiles seguintes adaptem suas análises automaticamente aos diferentes tipos de dados existentes.

# PERGUNTA RESPONDIDA:
# "Qual é a estrutura técnica do DataFrame?"

schema_info = []

for position, field in enumerate(df.schema.fields):
    schema_info.append({
        "position": position,
        "column": field.name,
        "type": field.dataType.simpleString(),
        "detailed_type": field.dataType.typeName(),
        "nullable": field.nullable
    })

schema_df = spark.createDataFrame(schema_info)

display(schema_df)

In [0]:
# ============================================================
# 02.5 CLASSIFICAÇÃO DOS TIPOS DE COLUNA
# ============================================================

# O QUE FAZ:
# Classifica automaticamente cada coluna em uma categoria analítica padronizada para utilização pelos Profiles.

# COMO FAZ:
# Utiliza os tipos nativos do Spark para agrupar as colunas em categorias como numérica, texto, data, data/hora e booleana.

# POR QUE É IMPORTANTE:
# Permite que cada Profile identifique automaticamente quais análises são aplicáveis a cada coluna sem depender do nome ou da origem dos dados.

# PERGUNTA RESPONDIDA:
# "Que tipo de análise pode ser aplicada a cada coluna?"

numeric_columns = []
text_columns = []
date_columns = []
datetime_columns = []
boolean_columns = []
others_columns = []

for field in df.schema.fields:
    if isinstance(field.dataType, NumericType):
        numeric_columns.append(field.name)
    elif isinstance(field.dataType, StringType):
        text_columns.append(field.name)
    elif isinstance(field.dataType, DateType):
        date_columns.append(field.name)
    elif isinstance(field.dataType, TimestampType):
        datetime_columns.append(field.name)
    elif isinstance(field.dataType, BooleanType):
        boolean_columns.append(field.name)
    else:
        others_columns.append(field.name)

print("CLASSIFICATION")
print("-" * 50)

print(f"Numeric columns: {len(numeric_columns)}")
print(f"Text columns: {len(text_columns)}")
print(f"Date columns: {len(date_columns)}")
print(f"DateTime columns: {len(datetime_columns)}")
print(f"Boolean columns: {len(boolean_columns)}")
print(f"Other columns: {len(others_columns)}")

In [0]:
# ============================================================
# 02.6 IDENTIFICAÇÃO DAS COLUNAS ANALISÁVEIS
# ============================================================

# O QUE FAZ:
# Define automaticamente quais Profiles podem ser aplicados a cada categoria de coluna.

# COMO FAZ:
# Utiliza a classificação dos tipos de dados criada anteriormente para construir listas de aplicabilidade analítica.

# POR QUE É IMPORTANTE:
# Evita análises inadequadas e permite que o framework seja genérico para diferentes estruturas de dados.

# PERGUNTA RESPONDIDA:
# "Quais análises podem ser executadas para cada tipo de coluna?"

columns_distribuition = (numeric_columns + text_columns + date_columns, datetime_columns, boolean_columns, others_columns)

columns_pattern = (text_columns)
columns_outlier = (numeric_columns)
columns_correlation = (numeric_columns)
columns_schema = (df.columns)


print("Aplicabilidade dos profiles")
print("-" * 50)

print(f"Schema Profile: {len(columns_schema)}")
print(f"Distribuition Profile: {len(columns_distribuition)}")
print(f"Pattern Profile: {len(columns_pattern)}")
print(f"Outlier Profile: {len(columns_outlier)}")
print(f"Correlation Profile: {len(columns_correlation)}")

In [0]:

# ============================================================
# 02.7 CONFIGURAÇÕES DO FRAMEWORK
# ============================================================

# O QUE FAZ:
# Centraliza os parâmetros utilizados pelos diferentes Profiles do framework de Data Quality.

# COMO FAZ:
# Define valores de configuração em um único local para que os Profiles possam reutilizá-los sem duplicação de parâmetros.

# POR QUE É IMPORTANTE:
# Facilita a manutenção, padroniza as análises e evita que diferentes Profiles utilizem parâmetros inconsistentes.

# PERGUNTA RESPONDIDA:
# "Quais parâmetros controlam o comportamento do framework?"

# ============================================================
# CONFIGS
# ============================================================

FRAMEWORK_NAME = "DATA QUALITY FRAMEWORK"

# ============================================================
# DISTRIBUTION PROFILE
# ============================================================

TOP_N = 5

TOP_CONCENTRATION_N = [1, 3, 5]

# ============================================================
# PATTERN PROFILE
# ============================================================

PATTERN_TOP_N = 5

# ============================================================
# DATA QUALITY SCORE
# ============================================================

SCORE_MIN = 0
SCORE_MAX = 100

# ============================================================
# EXECUÇÃO
# ============================================================

DISPLAY_RESULTS = True

print("Framework configs loaded.")


In [0]:
# ============================================================
# 02.8 DATAFRAME PREPARADO
# ============================================================

# O QUE FAZ:
# Define o DataFrame preparado que será utilizado como entrada pelos diferentes Profiles de qualidade.

# COMO FAZ:
# Mantém o DataFrame original da ingestão como base e cria uma referência padronizada para as etapas analíticas.

# POR QUE É IMPORTANTE:
# Estabelece um contrato único entre o Data Preparation e os Profiles seguintes, reduzindo a necessidade de repetir preparações técnicas.

# PERGUNTA RESPONDIDA:
# "Qual DataFrame será utilizado pelo framework de qualidade?"

df_prepared = df

print("DataFrame successfully prepared.")
print(f"Registers: {total_registers}")
print(f"Columns: {len(df_prepared.columns)}")